In [0]:
%run "/Workspace/Optum-DBx/Project-Optum-DBx/Dbx-Transformations/Connectors"

In [0]:
%run "/Workspace/Optum-DBx/Project-Optum-DBx/Dbx-Transformations/Generic"

In [0]:
#Importing neccessary libraries
from pyspark.sql.functions import *

In [0]:
#Calling the function to connect to ADLS storage from Connectors
adls_connect()

In [0]:
#Listing all the files in Silver layer
list_silver_files()

In [0]:
#Reading a csv file in Silver layer
grp = read_silver_file_csv("group_S")
subgrp = read_silver_file_csv("subgroup_S")
claims = read_silver_file_csv("claims_S")
disease = read_silver_file_csv("disease_S")
hospital = read_silver_file_csv("Hospital_S")
patient = read_silver_file_csv("patient_S")
subscriber = read_silver_file_csv("subscriber_S")

Joining all dataframes into a final dataframe (Warehouse)

In [0]:
grp_cols = [col(f"g.{c}").alias(f"grp_{c}") for c in grp.columns]
subgrp_cols = [col(f"sg.{c}").alias(f"subgrp_{c}") for c in subgrp.columns]
subscriber_cols = [col(f"s.{c}").alias(f"subscriber_{c}") for c in subscriber.columns]
claims_cols = [col(f"c.{c}").alias(f"claims_{c}") for c in claims.columns]
patient_cols = [col(f"p.{c}").alias(f"patient_{c}") for c in patient.columns]
hospital_cols = [col(f"h.{c}").alias(f"hospital_{c}") for c in hospital.columns]
disease_cols = [col(f"d.{c}").alias(f"disease_{c}") for c in disease.columns]

final_df = grp.alias("g") \
    .join(subgrp.alias("sg"), col("g.grp_id") == col("sg.subgrp_id"), "left") \
    .join(subscriber.alias("s"), col("sg.subgrp_sk") == col("s.Subgrp_id"), "left") \
    .join(claims.alias("c"), col("s.sub_id") == col("c.SUB_ID"), "left") \
    .join(patient.alias("p"), col("c.disease_name") == col("p.disease_name"), "left") \
    .join(hospital.alias("h"), col("p.hospital_id") == col("h.hospital_id"), "left") \
    .join(disease.alias("d"), col("sg.subgrp_sk") == col("d.subgrp_id"), "left") \
    .select(
        *(grp_cols +
          subgrp_cols +
          subscriber_cols +
          claims_cols +
          patient_cols +
          hospital_cols +
          disease_cols)
    )


In [0]:
display(final_df.limit(5))

In [0]:
#grp_subgrp_df = grp.join(subgrp, grp.grp_id == subgrp.subgrp_id, "left")

Writing to Gold layer

In [0]:
#Load final Dataframe into Gold layer and Azure SQL Database
write_to_gold(final_df, "Optum_G.csv")
write_to_database(final_df, "Optum_Tb")